<a href="https://colab.research.google.com/github/Anaemos/Aryavart-portfolio/blob/main/Emotional_Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import datasets
import transformers
import torch
import sklearn
import pyarrow

print("All libraries loaded successfully 🎉")


In [ ]:
with open("/kaggle/input/huggingfaceemotiondata/emotion", "r") as file:
    for i, line in enumerate(file):
        print(line.strip())
        if i == 10:  # Stop after 10 lines to avoid flooding
            break

In [ ]:
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("dair-ai/emotion")
train_data = dataset["train"]
print(train_data[:5])

In [ ]:
from datasets import DatasetDict

def add_emotion(example):
    emotion_labels = {0: "sadness", 1: "joy", 2: "love", 3: "anger", 4: "fear", 5: "surprise"}
    example["emotion"] = emotion_labels[example["label"]]
    return example

# Apply to all splits
dataset = dataset.map(add_emotion)
print(dataset["train"][:5])

In [ ]:
from transformers import AutoTokenizer

# Load a pre-trained tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# Apply the tokenizer to the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Display the tokenized dataset
print(tokenized_datasets)


In [ ]:
!pip install evaluate




In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
import evaluate
from datasets import load_dataset
import torch

# Confirm device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Load model — 6 emotion labels
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=6).to(device)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Load accuracy metric
accuracy = evaluate.load("accuracy")

# Define compute_metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

# Load dataset (Example: "emotion" dataset)
dataset = load_dataset("emotion")

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    disable_tqdm=True,
    report_to=None,
    logging_steps=10,
    run_name="emotion_classification_run",  # specify a different run name
)


# Setup Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

# Start training
trainer.train()
